# ASR Hutsul — Inference Demo

This notebook loads any of the four model families produced by
this project and transcribes a single audio file.

Supported checkpoints:

* Whisper (with or without PEFT/LoRA adapters)
* Wav2Vec2-XLSR (CTC)
* Wav2Vec2-BERT-UK (CTC, adapter-tuned)
* OmniASR (CTC, custom remote code)

The model family is auto-detected from the checkpoint's
`config.json` / `adapter_config.json`.

## 1. Setup

If you are running this notebook outside of the project repo,
install dependencies first:

```bash
pip install -r requirements.txt
```

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Allow ``import config`` and friends regardless of where the
# notebook is launched from.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'config.py').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

In [ ]:
import logging
import numpy as np
import torch

from config import configure_logging
from evaluate import (
    InferenceConfig,
    load_model_and_processor,
    run_inference,
)
from utils.text_normalization import build_default_normalizer

configure_logging(level=logging.INFO)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Pick a checkpoint

Set `CHECKPOINT_DIR` to the directory containing your trained
model.  This can be:

* `outputs/whisper/whisper-small/final` (full model)
* `outputs/whisper/whisper-small/checkpoint-2000` (intermediate)
* `outputs/whisper/whisper-small/final` with `adapter_config.json`
  (PEFT/LoRA adapters)
* a Hugging Face Hub model id (downloaded on demand)

In [ ]:
CHECKPOINT_DIR = Path('outputs/whisper/whisper-small/final')
MODEL_TYPE = None  # auto-detect; set to 'whisper'/'wav2vec2'/'wav2vec2_bert'/'omniasr' to override
HF_TOKEN = None    # paste an HF token here if the base model is gated

loaded = load_model_and_processor(
    CHECKPOINT_DIR,
    model_family=MODEL_TYPE,
    device=device,
    hf_token=HF_TOKEN,
)
print('Family :', loaded.family)
print('PEFT   :', loaded.is_peft)
print('Base id:', loaded.base_model_id)

## 3. Upload audio (optional)

On Colab the ``files.upload()`` widget shows a file picker.  When
running locally just point ``AUDIO_PATH`` to a WAV/FLAC/MP3 file.

In [ ]:
AUDIO_PATH: Path | None = None

try:
    from google.colab import files  # type: ignore
    print('Detected Colab — please upload an audio file:')
    uploaded = files.upload()
    if uploaded:
        AUDIO_PATH = Path(next(iter(uploaded.keys())))
        print('Uploaded:', AUDIO_PATH)
except ImportError:
    pass

if AUDIO_PATH is None:
    AUDIO_PATH = Path('example.wav')
    print('Set AUDIO_PATH to your file path:', AUDIO_PATH)

## 4. Transcribe

In [ ]:
import librosa

TARGET_SR = 16_000

if not AUDIO_PATH.exists():
    raise FileNotFoundError(f'AUDIO_PATH does not exist: {AUDIO_PATH}')

samples, sr = librosa.load(str(AUDIO_PATH), sr=TARGET_SR, mono=True)
samples = samples.astype(np.float32)
print(f'Loaded {len(samples)/TARGET_SR:.2f} s @ {TARGET_SR} Hz')

In [ ]:
fe = loaded.feature_extractor
batch = fe([samples], sampling_rate=TARGET_SR, return_tensors='pt', padding=True)
batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}

loaded.model.eval()
with torch.no_grad():
    if loaded.family == 'whisper':
        pred_ids = loaded.model.generate(
            **batch,
            max_new_tokens=225,
            num_beams=1,
            language='uk',
            task='transcribe',
        )
    else:
        outputs = loaded.model(**batch)
        pred_ids = outputs.logits.argmax(dim=-1)

raw = loaded.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)[0]
norm = build_default_normalizer()(raw)
print('Raw       :', raw)
print('Normalised:', norm)

## 5. Run on the project test split (optional)

Reproduces the WER / CER reported in the paper for the loaded
checkpoint.

In [ ]:
from config import ProjectConfig
from preprocess import load_and_prepare
from metrics import MetricCalculator, analyze_substitutions

project_cfg = ProjectConfig()
project_cfg.ensure_dirs()

dataset, audio_col, text_col = load_and_prepare(project_cfg, token=HF_TOKEN)
test_split = dataset['test']

predictions, references = run_inference(
    loaded,
    test_split,
    audio_column=audio_col,
    text_column=text_col,
    sample_rate=project_cfg.sample_rate,
    cfg=InferenceConfig(batch_size=8),
    device=device,
)

mc = MetricCalculator()
wer = mc.compute_wer(predictions, references)
cer = mc.compute_cer(predictions, references)
print(f'WER = {wer:.4f}')
print(f'CER = {cer:.4f}')

errors = analyze_substitutions(predictions, references)
print('Top substitutions:')
for entry in errors.to_dict(top_k=10)['top_substitutions']:
    print(f'  {entry["ref"]!r:>4} -> {entry["hyp"]!r:<4}  (n={entry["count"]})')